# Binary: colour features (lesion vs skin) on the preprocessed images

**Nevus vs others** using **colour features only**. Shape and texture come later.

* Input: `mini_dataset_preprocessed/` (dark borders removed → Shades of Gray → hair removal, from notebook 05).
* Each image is segmented (`segment_lesion`, mask branch) into **lesion, border ring and surrounding skin** (`region_masks`).
* Features (`feature_extraction/color_features.py → color_features`):

| group | features |
|---|---|
| `les`, `skin` | mean / std in Lab, RGB, HSV (hue as circular mean) of the lesion and of the skin |
| `mom` | skewness + entropy of the lesion (Lab, HSV) |
| `hist` | lesion histograms, HSV + Lab, 16 bins per channel |
| `dc` | fraction of the **6 dermoscopic colours** (white, red, light/dark brown, blue-gray, black) + number of colours |
| `var` | colour variegation inside the lesion (spread of ΔE) |
| `con` | contrast lesion − skin (ΔL, Δa, Δb, ΔE, RGB ratios) and border ring vs core |
| `seg` | segmentation area and fallback flag |

Models: Logistic Regression, SVM (RBF) and Random Forest, with 5-fold CV on **train**. `val` is used only to report results.

In [1]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name in ("Binary", "Multiclass", "notebooks") else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from feature_extraction import extract_color_image, select_groups, FEATURE_GROUPS

SEED = 42

ImportError: cannot import name 'extract_color_image' from 'feature_extraction' (C:\Users\Usuario\PycharmProjects\Melanoma-Challenge-ML-UdG\feature_extraction\__init__.py)

## 0. Configuration

In [ ]:
DATA_DIR = PROJECT_ROOT / "mini_dataset_preprocessed"   # output of notebook 05
USE_MASK = True            # False = whole image as "lesion" (no segmentation), to compare
N_JOBS = -1                # parallel extraction (-1 = all cores)

# feature groups used by the models (remove some to test them; see section 4b)
GROUPS = ["les", "skin", "mom", "hist", "dc", "var", "con", "seg"]

CACHE_DIR = PROJECT_ROOT / "Binary" / "features_cache"
CACHE_DIR.mkdir(exist_ok=True)

## 1. Image list and labels

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

def list_split(split):
    rows = []
    for cls_dir in sorted(p for p in (DATA_DIR / split).iterdir() if p.is_dir()):
        for f in sorted(f for f in cls_dir.rglob("*") if f.suffix.lower() in IMG_EXT):
            rows.append({"relpath": f.relative_to(DATA_DIR).as_posix(), "class": cls_dir.name.lower()})
    return pd.DataFrame(rows)

train_df, val_df = list_split("train"), list_split("val")
NEG = "nevus"
for d in (train_df, val_df):
    d["y"] = (d["class"] != NEG).astype(int)      # nevus = 0, others = 1
LABELS = [NEG, "others"]
pd.concat({"train": train_df["class"].value_counts(), "val": val_df["class"].value_counts()}, axis=1)

## 2. Feature extraction (cached)
The features are saved to `Binary/features_cache/color_v2_<split>_mask<0|1>.csv`, so re-running is instant. If the cache is incomplete, only the missing images are extracted. Delete the file to extract again, for example after changing the preprocessing or the segmentation.

In [ ]:
def get_features(df, split):
    f = CACHE_DIR / f"color_v2_{split}_mask{int(USE_MASK)}.csv"
    cached = pd.read_csv(f) if f.exists() else pd.DataFrame(columns=["relpath"])
    todo = df.loc[~df["relpath"].isin(cached["relpath"]), "relpath"].tolist()
    if todo:
        t0 = time.time()
        rows = Parallel(n_jobs=N_JOBS, batch_size=16)(
            delayed(extract_color_image)(DATA_DIR / r, USE_MASK) for r in todo)
        new = pd.DataFrame(rows).assign(relpath=todo)
        cached = pd.concat([cached, new], ignore_index=True)
        cached.to_csv(f, index=False)
        print(f"[extracted] {split}: {len(todo)} images in {time.time() - t0:.0f}s -> {f.name}")
    else:
        print(f"[cache] {f.name}")
    feats = df[["relpath"]].merge(cached, on="relpath", how="left")    # same order as df
    return feats.drop(columns="relpath")

F_train, F_val = get_features(train_df, "train"), get_features(val_df, "val")
y_train, y_val = train_df["y"].values, val_df["y"].values
print("all features:", F_train.shape[1], "| segmentation fallback (train):", f"{F_train['seg_fallback'].mean():.1%}")

### 2b. Which features separate the classes? (train only)
Effect size per feature (Cohen's d, others − nevus). The largest |d| are the most discriminative on their own.

In [ ]:
d = ((F_train[y_train == 1].mean() - F_train[y_train == 0].mean()) / (F_train.std() + 1e-9)).dropna()
top = d.reindex(d.abs().sort_values(ascending=False).index)[:25]
fig, ax = plt.subplots(figsize=(7, 7))
ax.barh(top.index[::-1], top.values[::-1], color=np.where(top.values[::-1] > 0, "#DD8452", "#4C72B0"))
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("Cohen's d  (orange: higher in 'others', blue: higher in nevus)")
plt.tight_layout(); plt.show()

## 3. Models

In [ ]:
X_train = select_groups(F_train, GROUPS).values
X_val = select_groups(F_val, GROUPS).values
print("features used:", X_train.shape[1], "| groups:", GROUPS)

cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
pre = [VarianceThreshold(1e-8), StandardScaler()]
models = {
    "LogReg": GridSearchCV(make_pipeline(*pre, LogisticRegression(max_iter=5000)),
                           {"logisticregression__C": [0.01, 0.1, 1]},
                           cv=cv, scoring="balanced_accuracy", n_jobs=N_JOBS),
    "SVM-RBF": GridSearchCV(make_pipeline(*pre, SVC(probability=True, random_state=SEED)),
                            {"svc__C": [1, 10], "svc__gamma": ["scale", 0.001]},
                            cv=cv, scoring="balanced_accuracy", n_jobs=N_JOBS),
    "RandomForest": GridSearchCV(make_pipeline(VarianceThreshold(1e-8),
                                               RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=N_JOBS)),
                                 {"randomforestclassifier__max_features": ["sqrt", 0.2],
                                  "randomforestclassifier__min_samples_leaf": [1, 3]},
                                 cv=cv, scoring="balanced_accuracy", n_jobs=1),
}

def evaluate(gs, Xv):
    pred, prob = gs.predict(Xv), gs.predict_proba(Xv)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    return {"cv_bal_acc (train)": round(gs.best_score_, 4),
            "val_accuracy": round(accuracy_score(y_val, pred), 4),
            "val_bal_accuracy": round(balanced_accuracy_score(y_val, pred), 4),
            "val_sensitivity": round(tp / (tp + fn), 4),
            "val_specificity": round(tn / (tn + fp), 4),
            "val_auc": round(roc_auc_score(y_val, prob), 4)}

results = []
for name, gs in models.items():
    t0 = time.time()
    gs.fit(X_train, y_train)
    results.append({"model": name, "best_params": gs.best_params_, **evaluate(gs, X_val),
                    "fit_time_s": round(time.time() - t0, 1)})
res = pd.DataFrame(results).set_index("model")
res

## 4. Confusion matrices and feature importance

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(4.5 * len(models), 4))
for ax, (name, gs) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_predictions(y_val, gs.predict(X_val), display_labels=LABELS,
                                            cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(f"{name} (bal. acc {res.loc[name, 'val_bal_accuracy']:.3f})")
plt.tight_layout(); plt.show()

rf = models["RandomForest"].best_estimator_
names = select_groups(F_train, GROUPS).columns[rf[0].get_support()]
imp = pd.Series(rf[-1].feature_importances_, index=names).sort_values(ascending=False)
print("importance per group:")
print(imp.groupby(imp.index.str.split("_").str[0]).sum().sort_values(ascending=False).round(3))
imp[:20][::-1].plot.barh(figsize=(6, 6), title="Random Forest: top 20 features"); plt.tight_layout(); plt.show()

### 4b. Ablation: what does each group add? (LogReg, fast)
Each row trains a model **without** one group. A big drop means the group matters.

In [ ]:
abl = []
for g in [None] + GROUPS:
    use = [x for x in GROUPS if x != g]
    gs = GridSearchCV(make_pipeline(*pre, LogisticRegression(max_iter=5000)),
                      {"logisticregression__C": [0.01, 0.1, 1]}, cv=cv, scoring="balanced_accuracy", n_jobs=N_JOBS)
    gs.fit(select_groups(F_train, use).values, y_train)
    abl.append({"without": g or "(all)", "n_features": len(select_groups(F_train, use).columns),
                **evaluate(gs, select_groups(F_val, use).values)})
pd.DataFrame(abl).set_index("without")

## 5. Save results

In [ ]:
exp_name = f"color_v2_mask{int(USE_MASK)}_{'-'.join(GROUPS)}"
out = PROJECT_ROOT / "Binary" / "results_baseline.csv"
row = res.reset_index().assign(experiment=exp_name, preprocess="05_explorer", feature="color_v2",
                               n_features=X_train.shape[1])
row["best_params"] = row["best_params"].astype(str)
if out.exists():
    old = pd.read_csv(out)
    row = pd.concat([old[old["experiment"] != exp_name], row], ignore_index=True)
row.to_csv(out, index=False)
print("saved ->", out)